# OpenClip on Kaggle
Settings → **Accelerator: GPU T4 x2** and **Internet: On**.
Secrets (Add-ons → Secrets): `LLM_BASE_URL` (your gateway's **tunnel** URL, e.g. `https://xxx.trycloudflare.com` — 127.0.0.1 is your PC, Kaggle can't reach it), `LLM_API_KEY`; optional `GEMINI_API_KEY`, `YOUTUBE_COOKIES` (Netscape format).
Run all cells; the last one prints a public URL for the dashboard.

In [ ]:
import os, subprocess, urllib.request
from kaggle_secrets import UserSecretsClient
def secret(name):
    try: return UserSecretsClient().get_secret(name)
    except Exception: return None

# ---- LLM gateway (picks the clip moments). Optional here: you can also set
# URL / key / model later from the dashboard popup or Settings.
# Kaggle runs in the cloud, so 127.0.0.1 here is Kaggle itself, not your PC.
# Expose your gateway first, on your PC:
#   cloudflared tunnel --url http://127.0.0.1:47821
# and use the https://....trycloudflare.com URL it prints.
PUBLIC_URL = secret('LLM_BASE_URL') or ""   # e.g. "https://xxxx.trycloudflare.com"
API_KEY = secret('LLM_API_KEY') or ""      # keep it in a Kaggle secret, not in this cell
MODEL = "google/gemini-3.8-flash"

# ---- Ollama on this Kaggle box instead (runs on the T4s next to the jobs, no
# tunnel or key needed). True = install it, pull the model, and use it.
USE_OLLAMA = False
OLLAMA_MODEL = "qwen2.5:7b"   # any tag from https://ollama.com/library that fits ~8 GB

# GPU split. With Ollama on and two T4s: the last card (GPU 1) is Ollama's
# alone, the jobs and the server get the rest (GPU 0). Otherwise the jobs
# take turns on every card.
N_GPUS = len(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout.splitlines())
OLLAMA_GPU = str(N_GPUS - 1) if USE_OLLAMA and N_GPUS > 1 else None
JOB_GPUS = [str(i) for i in range(N_GPUS) if str(i) != OLLAMA_GPU]
print(f'GPUs: {N_GPUS} | jobs on {",".join(JOB_GPUS) or "CPU"}' + (f' | Ollama on {OLLAMA_GPU}' if OLLAMA_GPU else ''))

def gateway_status(url, key):
    """HTTP status of <url>/v1/models, or the error text."""
    base = url.rstrip('/').removesuffix('/v1') + '/v1'
    req = urllib.request.Request(base + '/models', headers={'Authorization': f'Bearer {key or "x"}'})
    try:
        return urllib.request.urlopen(req, timeout=15).status
    except urllib.error.HTTPError as e:
        return e.code
    except Exception as e:
        return str(e)[:120]

if not PUBLIC_URL:
    print('ℹ️ No gateway here: set it in the dashboard popup / Settings, or give a GEMINI_API_KEY.')
elif any(h in PUBLIC_URL for h in ('127.0.0.1', 'localhost', '0.0.0.0')):
    print(f'❌ {PUBLIC_URL} is Kaggle itself, not your PC. Use the tunnel URL instead.')
    PUBLIC_URL = ""
else:
    st = gateway_status(PUBLIC_URL, API_KEY)
    ok = st == 200
    print(('✅' if ok else '❌'), 'LLM gateway', PUBLIC_URL, '→', st, '|', MODEL, '| key set' if API_KEY else '| NO KEY')
    if st == 401: print('   The URL works but the key was refused: check LLM_API_KEY.')


In [ ]:
import os, subprocess
os.chdir('/kaggle/working')
if not os.path.exists('OpenClip'):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/sanupyadav/OpenClip.git'], check=True)
else:
    subprocess.run(['git', '-C', 'OpenClip', 'pull', '--ff-only'], check=True)
os.chdir('/kaggle/working/OpenClip')
!git log --oneline -1

In [ ]:
%%bash
set -e
apt-get update -qq
apt-get install -y -qq ffmpeg fontconfig fonts-liberation fonts-noto-color-emoji > /dev/null
# Node 20 for the dashboard
if ! node -v 2>/dev/null | grep -q '^v2[0-9]'; then
  curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /dev/null
  apt-get install -y -qq nodejs > /dev/null
fi
# Bundled fonts for libass (same as the Dockerfile)
mkdir -p /usr/local/share/fonts/openshorts
cp fonts/*.ttf /usr/local/share/fonts/openshorts/
cp fonts/openshorts-fontmap.conf /etc/fonts/conf.d/60-openshorts.conf
fc-cache -f > /dev/null
# cloudflared quick tunnel
[ -x /usr/local/bin/cloudflared ] || curl -fsSL -o /usr/local/bin/cloudflared \
  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared
node -v; ffmpeg -version | head -1

In [ ]:
%%bash
set -e
# Own venv on Python 3.11, like the Docker image: Kaggle's preinstalled packages
# need protobuf 5, mediapipe 0.10.14 needs protobuf 4, and they cannot share one env.
pip install -q uv
[ -x /kaggle/venv/bin/python ] || uv venv -q --python 3.11 /kaggle/venv
# Kaggle presets uv to the system Python, so name the venv on every install.
unset UV_SYSTEM_PYTHON UV_PYTHON
# The notebook's inline matplotlib backend and PYTHONPATH do not exist in the venv.
export MPLBACKEND=Agg
unset PYTHONPATH
UVP="uv pip install -q --python /kaggle/venv/bin/python"
# torch for the CUDA the T4 driver supports (the pinned wheel may need a newer one)
$UVP torch torchvision --torch-backend=auto
grep -vE '^(torch|torchvision)==' requirements.txt > /tmp/req.txt
$UVP -r /tmp/req.txt
$UVP "nvidia-cublas-cu12<13" "nvidia-cudnn-cu12>=9,<10" onnx-asr onnxruntime-gpu
$UVP --upgrade --prerelease=allow "yt-dlp[default]"
/kaggle/venv/bin/python -c "import torch, mediapipe, google.protobuf as p; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), '| protobuf', p.__version__)"
cd dashboard && npm install --no-audit --no-fund --loglevel=error


In [ ]:
# Ollama (only when USE_OLLAMA = True in the config cell).
import time, urllib.request
if USE_OLLAMA:
    if subprocess.run(['which', 'ollama'], capture_output=True).returncode != 0:
        subprocess.run('apt-get install -y -qq zstd > /dev/null; curl -fsSL https://ollama.com/install.sh | sh',
                       shell=True, check=True)
    subprocess.run(['pkill', '-f', 'ollama serve'])
    ollama_env = dict(os.environ, OLLAMA_HOST='127.0.0.1:11434', CUDA_DEVICE_ORDER='PCI_BUS_ID')
    if OLLAMA_GPU is not None:
        ollama_env['CUDA_VISIBLE_DEVICES'] = OLLAMA_GPU  # its own card, away from the jobs
    ollama = subprocess.Popen(['ollama', 'serve'], env=ollama_env,
                              stdout=open('ollama.log', 'w'), stderr=subprocess.STDOUT)
    for _ in range(60):
        try:
            urllib.request.urlopen('http://127.0.0.1:11434/api/tags', timeout=2); break
        except Exception:
            time.sleep(1)
    subprocess.run(['ollama', 'pull', OLLAMA_MODEL], check=True)
    print('✅ Ollama ready with', OLLAMA_MODEL, f'on GPU {OLLAMA_GPU}' if OLLAMA_GPU else '')
else:
    print('Ollama off (USE_OLLAMA = False).')


In [ ]:
import re, pathlib, time, json, glob, urllib.request
# Let the tunnel hostname through Vite's host check.
cfg = pathlib.Path('dashboard/vite.config.js')
cfg.write_text(re.sub(r'allowedHosts:\s*\[[^\]]*\]', 'allowedHosts: true', cfg.read_text()))

VENV = '/kaggle/venv'
site = f'{VENV}/lib/python3.11/site-packages'
env = dict(os.environ, VIRTUAL_ENV=VENV, PATH=f'{VENV}/bin:' + os.environ['PATH'],
    PYTHONUNBUFFERED='1',
    WHISPER_MODEL='large-v3-turbo', WHISPER_DEVICE='cuda', WHISPER_COMPUTE='float16',
    FFMPEG_ENCODER='nvenc', MAX_CONCURRENT_JOBS='2',  # one per T4, or both on GPU 0 when Ollama has GPU 1
    LD_LIBRARY_PATH=f'{site}/nvidia/cublas/lib:{site}/nvidia/cudnn/lib:' + os.environ.get('LD_LIBRARY_PATH', ''),
)
# The job cards, for the wrapper below; the server's own GPU work (thumbnail
# and dubbed-subtitle transcription) stays on them too, never on Ollama's.
env['JOB_GPUS'] = ','.join(JOB_GPUS)
env['CUDA_DEVICE_ORDER'] = 'PCI_BUS_ID'
if JOB_GPUS:
    env['CUDA_VISIBLE_DEVICES'] = ','.join(JOB_GPUS)
env['MPLBACKEND'] = 'Agg'  # the notebook's inline backend is not in the venv
env.pop('PYTHONPATH', None)
if PUBLIC_URL and not USE_OLLAMA:
    env['LLM_SCORE_BATCH'] = '8'  # a hosted model takes 8 windows a call; Ollama keeps the default 3
for k in ('GEMINI_API_KEY', 'YOUTUBE_COOKIES'):
    if (v := secret(k)): env[k] = v

os.makedirs('output', exist_ok=True)

# Fresh start: stop whatever an earlier run of this cell left behind (server,
# running jobs, dashboard, tunnel) and drop its queue, so nothing old resumes.
# One pkill per pattern and no shell: a `sh -c "pkill ...; pkill ..."` has
# every pattern in its own command line and the first pkill kills it.
for pat in ('uvicorn app:app', 'main.py', 'vite', 'cloudflared tunnel'):
    subprocess.run(['pkill', '-f', pat])
for m in glob.glob('output/*/.resume.json'):
    os.remove(m)
time.sleep(2)

# app.py runs each job as `sys.executable -u main.py`, so the server starts
# through this wrapper and every job takes the next card of JOB_GPUS.
# ponytail: round-robin, not by load; fine at MAX_CONCURRENT_JOBS=2.
wrapper = f'{VENV}/bin/python-gpu'
open(wrapper, 'w').write(r"""#!/bin/bash
case " $* " in *" main.py "*)
  IFS=, read -ra G <<< "${JOB_GPUS:-}"
  n=${#G[@]}
  if [ "$n" -gt 0 ]; then
    i=$(flock /tmp/gpu-rr.lock sh -c 'i=$(cat /tmp/gpu-rr 2>/dev/null || echo 0); echo $(( (i+1) % '"$n"' )) > /tmp/gpu-rr; echo $(( i % '"$n"' ))')
    export CUDA_VISIBLE_DEVICES=${G[$i]} CUDA_DEVICE_ORDER=PCI_BUS_ID
    echo "GPU ${G[$i]}: $*" >> /kaggle/working/OpenClip/gpu.log
  fi;;
esac
exec -a "$0" "$(dirname "$0")/python3.11" "$@"
""")
os.chmod(wrapper, 0o755)
backend = subprocess.Popen([wrapper, '-m', 'uvicorn', 'app:app', '--host', '0.0.0.0', '--port', '8000'],
                           env=env, stdout=open('backend.log', 'w'), stderr=subprocess.STDOUT)
frontend = subprocess.Popen(['npm', 'run', 'dev', '--', '--host', '0.0.0.0', '--port', '5173'], cwd='dashboard',
                            env=dict(env, VITE_PROXY_TARGET='http://localhost:8000'),
                            stdout=open('frontend.log', 'w'), stderr=subprocess.STDOUT)
tunnel = subprocess.Popen(['cloudflared', 'tunnel', '--no-autoupdate', '--url', 'http://localhost:5173'],
                          stdout=open('tunnel.log', 'w'), stderr=subprocess.STDOUT)

def api(method, path, body=None):
    req = urllib.request.Request('http://localhost:8000' + path, method=method,
                                 data=json.dumps(body).encode() if body is not None else None,
                                 headers={'Content-Type': 'application/json'})
    return json.loads(urllib.request.urlopen(req, timeout=30).read())

for _ in range(90):
    try:
        api('GET', '/health'); break
    except Exception:
        time.sleep(2)

# The model settings go through the API (the same as Settings in the dashboard),
# so this cell wins over anything saved there in an earlier run.
if PUBLIC_URL:
    api('PUT', '/api/llm/config', {'base_url': PUBLIC_URL, 'api_key': API_KEY, 'model': MODEL})
if USE_OLLAMA:
    api('PUT', '/api/llm/ollama', {'url': 'http://127.0.0.1:11434', 'model': OLLAMA_MODEL, 'enabled': True})
llm = api('GET', '/api/config').get('localLlm')
print('AI model:', f"{llm['model']} @ {llm['baseUrl']}" if llm else 'none (Gemini key needed)')

for _ in range(60):
    time.sleep(2)
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('tunnel.log').read())
    if m: break
print('Dashboard:', m.group(0) if m else 'not ready, check tunnel.log')

In [ ]:
# Logs (re-run any time). Clips land in /kaggle/working/OpenClip/output.
!tail -n 30 backend.log; tail -n 10 frontend.log
# Which GPU each job got, and live usage of both cards
!cat gpu.log 2>/dev/null; nvidia-smi --query-gpu=index,utilization.gpu,memory.used --format=csv

In [ ]:
# Keep the session alive while you use the dashboard (interrupt to stop).
import time
while backend.poll() is None: time.sleep(60)
print('backend exited:', backend.returncode)